In [1]:
import gradio as gr
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import numpy as np

C:\Users\jhonn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DentalDetectionApp:
    def __init__(self, model_path):
        """
        Inicializar la aplicación de detección dental
        
        Args:
            model_path: Ruta al modelo entrenado (.pt)
        """
        print(f"Cargando modelo desde: {model_path}")
        self.model = YOLO(model_path)
        
        # Nombres de las clases
        self.class_names = {
            0: 'Caries',
            1: 'Diente Retenido',
            2: 'Pérdida Ósea'
        }
        
        # Colores para cada clase (RGB)
        self.class_colors = {
            0: (255, 0, 0),      # Rojo para Caries
            1: (0, 255, 0),      # Verde para Diente Retenido
            2: (0, 0, 255)       # Azul para Pérdida Ósea
        }
    
    def analyze_image(self, image, confidence_threshold=0.25):
        """
        Analizar una imagen dental y retornar resultados
        
        Args:
            image: Imagen PIL o numpy array
            confidence_threshold: Umbral mínimo de confianza (0-1)
            
        Returns:
            tuple: (imagen_con_detecciones, texto_resultados, estadísticas)
        """
        if image is None:
            return None, "Por favor, sube una imagen", ""
        
        # Convertir a PIL si es necesario
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        
        # Hacer predicción
        results = self.model.predict(
            source=image,
            conf=confidence_threshold,
            verbose=False
        )
        
        # Obtener el primer resultado
        result = results[0]
        
        # Extraer detecciones
        boxes = result.boxes
        
        if len(boxes) == 0:
            return image, "No se detectaron problemas dentales en esta imagen", ""
        
        # Crear copia de la imagen para dibujar
        img_draw = image.copy()
        draw = ImageDraw.Draw(img_draw)
        
        # Intentar cargar fuente
        try:
            font = ImageFont.truetype("arial.ttf", 20)
            font_small = ImageFont.truetype("arial.ttf", 16)
        except:
            font = ImageFont.load_default()
            font_small = ImageFont.load_default()
        
        # Contadores por clase
        class_counts = {0: 0, 1: 0, 2: 0}
        class_confidences = {0: [], 1: [], 2: []}
        
        detections_list = []
        
        # Procesar cada detección
        for box in boxes:
            # Extraer información
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            confidence = float(box.conf[0].cpu().numpy())
            class_id = int(box.cls[0].cpu().numpy())
            
            # Actualizar contadores
            class_counts[class_id] += 1
            class_confidences[class_id].append(confidence)
            
            # Nombre de la clase
            class_name = self.class_names[class_id]
            color = self.class_colors[class_id]
            
            # Dibujar bounding box
            draw.rectangle(
                [(x1, y1), (x2, y2)],
                outline=color,
                width=3
            )
            
            # Preparar etiqueta
            label = f"{class_name}: {confidence:.1%}"
            
            # Dibujar fondo para el texto
            bbox = draw.textbbox((x1, y1 - 25), label, font=font_small)
            draw.rectangle(bbox, fill=color)
            
            # Dibujar texto
            draw.text(
                (x1, y1 - 25),
                label,
                fill="white",
                font=font_small
            )
            
            # Guardar detección
            detections_list.append({
                'clase': class_name,
                'confianza': confidence,
                'posicion': (int(x1), int(y1), int(x2), int(y2))
            })
        
        # Generar reporte de texto
        report = self.generate_report(class_counts, class_confidences, detections_list)
        
        # Generar estadísticas
        stats = self.generate_statistics(class_counts, class_confidences)
        
        return img_draw, report, stats
    
    def generate_report(self, class_counts, class_confidences, detections_list):
        """
        Generar reporte de texto de las detecciones
        """
        report = "ANÁLISIS DE RADIOGRAFÍA DENTAL\n"
        report += "="*50 + "\n\n"
        
        total_detections = sum(class_counts.values())
        report += f"Total de detecciones: {total_detections}\n\n"
        
        # Resumen por clase
        for class_id, count in class_counts.items():
            if count > 0:
                class_name = self.class_names[class_id]
                avg_conf = np.mean(class_confidences[class_id])
                
                report += f"{class_name}:\n"
                report += f"  • Cantidad detectada: {count}\n"
                report += f"  • Confianza promedio: {avg_conf:.1%}\n"
                
                # Listar cada detección
                class_detections = [d for d in detections_list if d['clase'] == class_name]
                for i, det in enumerate(class_detections, 1):
                    report += f"  • Detección #{i}: {det['confianza']:.1%}\n"
                
                report += "\n"
        
        # Interpretación
        report += "INTERPRETACIÓN:\n"
        report += "-"*50 + "\n"
        
        if class_counts[0] > 0:
            report += f"⚠️ Se detectaron {class_counts[0]} caries\n"
        
        if class_counts[1] > 0:
            report += f"⚠️ Se detectaron {class_counts[1]} dientes retenidos\n"
        
        if class_counts[2] > 0:
            report += f"⚠️ Se detectó pérdida ósea en {class_counts[2]} zona(s)\n"
        
        if total_detections == 0:
            report += "✅ No se detectaron problemas significativos\n"
        
        report += "\nNOTA: Este análisis es una herramienta de apoyo.\n"
        report += "Consulte con un profesional para diagnóstico definitivo."
        
        return report
    
    def generate_statistics(self, class_counts, class_confidences):
        """
        Generar estadísticas en formato de tabla
        """
        stats = "ESTADÍSTICAS DETALLADAS\n"
        stats += "="*50 + "\n\n"
        
        stats += f"{'Condición':<20} {'Cantidad':<10} {'Confianza':<15}\n"
        stats += "-"*50 + "\n"
        
        for class_id in [0, 1, 2]:
            count = class_counts[class_id]
            class_name = self.class_names[class_id]
            
            if count > 0:
                avg_conf = np.mean(class_confidences[class_id])
                max_conf = max(class_confidences[class_id])
                min_conf = min(class_confidences[class_id])
                
                stats += f"{class_name:<20} {count:<10} Prom: {avg_conf:.1%}\n"
                stats += f"{'':<20} {'':<10} Max: {max_conf:.1%}\n"
                stats += f"{'':<20} {'':<10} Min: {min_conf:.1%}\n"
                stats += "\n"
            else:
                stats += f"{class_name:<20} {count:<10} N/A\n\n"
        
        return stats

def create_gradio_interface(model_path):
    """
    Crear la interfaz Gradio
    
    Args:
        model_path: Ruta al modelo entrenado
    """
    # Inicializar la aplicación
    app = DentalDetectionApp(model_path)
    
    # Crear interfaz
    with gr.Blocks(title="Detector de Problemas Dentales", theme=gr.themes.Soft()) as demo:
        
        gr.Markdown(
            """
            # 🦷 Sistema de Detección de Problemas Dentales
            
            Sube una radiografía dental para detectar automáticamente:
            - 🔴 **Caries**
            - 🟢 **Dientes Retenidos**
            - 🔵 **Pérdida Ósea**
            
            ---
            """
        )
        
        with gr.Row():
            with gr.Column(scale=1):
                # Input - SOLO subir archivos
                image_input = gr.Image(
                    type="pil",
                    label="Subir Radiografía Dental",
                    sources=["upload"],  # SOLO opción de subir archivos
                    height=400
                )
                
                confidence_slider = gr.Slider(
                    minimum=0.1,
                    maximum=0.9,
                    value=0.25,
                    step=0.05,
                    label="Umbral de Confianza",
                    info="Ajusta la sensibilidad de las detecciones"
                )
                
                analyze_button = gr.Button(
                    "🔍 Analizar Radiografía",
                    variant="primary",
                    size="lg"
                )
                
                clear_button = gr.Button(
                    "🗑️ Limpiar",
                    variant="secondary"
                )
            
            with gr.Column(scale=1):
                # Output
                image_output = gr.Image(
                    type="pil",
                    label="Resultado con Detecciones",
                    height=400
                )
        
        with gr.Row():
            with gr.Column():
                report_output = gr.Textbox(
                    label="📋 Reporte de Análisis",
                    lines=15,
                    max_lines=20
                )
            
            with gr.Column():
                stats_output = gr.Textbox(
                    label="📊 Estadísticas",
                    lines=15,
                    max_lines=20
                )
        
        # Ejemplos (opcional - puedes comentar si no tienes imágenes de ejemplo)
        gr.Markdown("### 📁 Ejemplos de uso")
        gr.Examples(
            examples=[
                # Puedes agregar rutas a imágenes de ejemplo aquí
                # ["ejemplo1.jpg", 0.25],
                # ["ejemplo2.jpg", 0.3],
            ],
            inputs=[image_input, confidence_slider],
            outputs=[image_output, report_output, stats_output],
            fn=app.analyze_image,
            cache_examples=False,
        )
        
        # Event handlers
        analyze_button.click(
            fn=app.analyze_image,
            inputs=[image_input, confidence_slider],
            outputs=[image_output, report_output, stats_output]
        )
        
        clear_button.click(
            fn=lambda: (None, None, "", ""),
            inputs=None,
            outputs=[image_input, image_output, report_output, stats_output]
        )
        
        gr.Markdown(
            """
            ---
            ### ⚠️ Advertencia Médica
            Este sistema es una herramienta de apoyo diagnóstico y **NO reemplaza** 
            la evaluación de un profesional de la salud dental calificado.
            
            **Precisión del modelo:** 79.0% (Fine-tuned con 9K+ imágenes)  
            **Recomendación:** Use este sistema solo como referencia preliminar.
            """
        )
    
    return demo

# Función principal para ejecutar
def main():
    """
    Función principal para ejecutar la aplicación
    """
    # CAMBIAR ESTA RUTA POR LA RUTA DE TU MODELO MEJORADO
    MODEL_PATH = r"C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\finetuning_results_9k\finetuned_9k\weights\best.pt"
    
    print("Iniciando aplicación Gradio...")
    print(f"Modelo: {MODEL_PATH}")
    print("Precisión del modelo: 79.0%")
    
    # Crear interfaz
    demo = create_gradio_interface(MODEL_PATH)
    
    # Lanzar aplicación
    demo.launch(
        share=False,        # Cambiar a True si quieres compartir públicamente
        server_name="0.0.0.0",  # Permite acceso desde otros dispositivos en la red
        server_port=7860,   # Puerto por defecto
        debug=True          # Modo debug para desarrollo
    )


In [3]:
if __name__ == "__main__":
    main()

Iniciando aplicación Gradio...
Modelo: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\finetuning_results_9k\finetuned_9k\weights\best.pt
Precisión del modelo: 79.0%
Cargando modelo desde: C:\Users\jhonn\OneDrive\Desktop\dataset_dientes\el_candidato\YOLO\YOLO\finetuning_results_9k\finetuned_9k\weights\best.pt
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
